In [4]:
import faiss 
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Sample documents
documents = [
    "Python is a versatile programming language used for web development and data science.",
    "Machine learning models require large amounts of training data to perform well.",
    "Neural networks are inspired by the structure of the human brain.",
    "Natural language processing enables computers to understand human language.",
    "Deep learning is a subset of machine learning using multi-layered neural networks.",
    "Data visualization helps communicate insights from complex datasets.",
    "Cloud computing provides on-demand access to computing resources.",
    "Cybersecurity protects systems and networks from digital attacks.",
    "Blockchain technology enables secure, decentralized transactions.",
    "Quantum computing uses quantum mechanics to solve complex problems."
]

In [40]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(documents)

In [41]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

In [42]:
index.add(embeddings)

In [43]:
query = ["What is artificial intelligence and machine learning"]
query_embedding = model.encode(query)
print(query_embedding.shape)

#Find top 3 most similar vectors
distances, indices = index.search(query_embedding, 3)

(1, 384)


In [12]:
for i, (idx, distance) in enumerate(zip(indices[0], distances[0]), 1):
    print(f"{i}. (Distance: {distance:.4f})")
    print(f"   {documents[idx]}")
    print()

1. (Distance: 0.9125)
   Deep learning is a subset of machine learning using multi-layered neural networks.

2. (Distance: 1.1951)
   Machine learning models require large amounts of training data to perform well.

3. (Distance: 1.2401)
   Natural language processing enables computers to understand human language.



### Cosine Similarity

$$S_c(x,y) = \frac{x.y}{||x|| \times ||y||}$$

In [13]:
#Normalize Embeddings
embeddings_norm = embeddings / np.linalg.norm(embeddings)
index_cosine_sim = faiss.IndexFlatIP(embeddings_norm.shape[1])
index_cosine_sim.add(embeddings_norm)
 
query_embedding_sim = query_embedding / np.linalg.norm(query_embedding)

distance_cos, index_cos = index_cosine_sim.search(query_embedding_sim, k = 3)
for i, (index, distance) in enumerate(zip(index_cos[0], distance_cos[0]), 1):
    print(f"{i}. (Distance: {distance:.4f})")
    print(f"   {documents[index]}")
    print()

1. (Distance: 0.1719)
   Deep learning is a subset of machine learning using multi-layered neural networks.

2. (Distance: 0.1273)
   Machine learning models require large amounts of training data to perform well.

3. (Distance: 0.1201)
   Natural language processing enables computers to understand human language.



In [14]:
#Persists to memory
faiss.write_index(index_cosine_sim, "faiss_index.bin")
print("Index saved to disk")

#Save documents
import pickle
with open("documents.pkl", "wb") as f:
    pickle.dump(documents, f)

#load documents
with open("documents.pkl", "rb") as f:
    loaded_documents = pickle.load(f) 

print("Loaded Documents Count: ", len(loaded_documents))

Index saved to disk
Loaded Documents Count:  10


### Exercise

* Take 1,000–10,000 text sentences (Wikipedia, news, StackOverflow)
* Generate embeddings (SentenceTransformers / OpenAI)
* Build:
- IndexFlatL2
- IndexIVFFlat
* Query with a sentence
* Return top-k similar sentences

In [ ]:
import wikipediaapi
from sentence_transformers import SentenceTransformer
import faiss
class Wikpedia:
    def __init__(self, topics: list):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.topics = topics
        self.index = None


    def search_wikipedia(self):
        summary_list = []
        wiki_tool = wikipediaapi.Wikipedia(user_agent=f"Teaching assistant ({"jofesdavid@gmail.com"})", language="en")
        for topic in self.topics:
            wiki_page_target = wiki_tool.page(topic) 
            if wiki_page_target.exists():
                summary = wiki_page_target.summary.split("/n")[0:5]
                summary = "\n".join(summary)
                summary_list.append(summary)
            else: 
                return summary_list.append("")
        return summary_list
        
    def create_embedding(self, documents, query: list):
        embeddings = self.model.encode(documents)
        query_embedding = self.model.encode(query)
        return query_embedding, embeddings
    
    def similarity_search(self, query_embedding, embeddings, k = 1):
        self.index = faiss.IndexFlatL2(embeddings.shape[1])
        self.index.add(embeddings)
        similarities, indices = self.index.search(query_embedding, k)
        faiss.write_index(self.index, "class_faiss_index.bin")
        return {"Distances": similarities, "Indices": indices}
    
    def top_k_results(self, indices, similarities, documents):
        for i, (index, similarity) in enumerate(zip(indices, similarities), 1):
            print(f"{i}. {documents[index]} \nSimilarity Score: {similarity}")

    






    

In [ ]:
topics = Wikpedia(topics=["calculus", "linear algebra"])
documents = topics.search_wikipedia()
query = ["What is differentiation?"]
query_embeddings, embeddings = topics.create_embedding(documents = documents, query=query)
similarity_dic = topics.similarity_search(query_embedding=query_embeddings, embeddings=embeddings)
results = topics.top_k_results(similarity_dic["Indices"][0], similarity_dic["Distances"][0], documents)

1. Calculus is the mathematical study of continuous change, in the same way that geometry is the study of shape, and algebra is the study of generalizations of arithmetic operations.
Originally called infinitesimal calculus or "the calculus of infinitesimals", it has two major branches, differential calculus and integral calculus. The former concerns instantaneous rates of change, and the slopes of curves, while the latter concerns accumulation of quantities, and areas under or between curves. These two branches are related to each other by the fundamental theorem of calculus. They make use of the fundamental notions of convergence of infinite sequences and infinite series to a well-defined limit. It is the "mathematical backbone" for dealing with problems where variables change with time or another reference variable. It has also been called "the basic instrument of physical science".
Infinitesimal calculus was formulated separately in the late 17th century by Isaac Newton and Gottfri

: 

In [6]:
client = chromadb.PersistentClient("./new_db")
collection = client.get_or_create_collection(name = "my_documents", metadata = {"description": "Sample data collection"})
print(f"Documents count: {collection.count()}")

Documents count: 0


In [8]:
# Sample documents with metadata
documents = [
    "Python is a versatile programming language used for web development and data science.",
    "Machine learning models require large amounts of training data to perform well.",
    "Neural networks are inspired by the structure of the human brain.",
    "Natural language processing enables computers to understand human language.",
    "Deep learning is a subset of machine learning using multi-layered neural networks."
]

# Metadata for each document
metadatas = [
    {"category": "programming", "topic": "python"},
    {"category": "AI", "topic": "machine learning"},
    {"category": "AI", "topic": "neural networks"},
    {"category": "AI", "topic": "NLP"},
    {"category": "AI", "topic": "deep learning"}
]

ids = [f"doc_{i}" for i in documents]
collection.add(documents=documents, metadatas=metadatas, ids=ids)

print(len(documents))
print(collection.count())

5
5


In [9]:
results = collection.query(
    query_texts = ["What is Artificial Intelligence"],
    n_results=3
)

for i, (doc, metadata, distance) in enumerate(zip(results["documents"][0],
                                                  results["metadatas"][0],
                                                  results["distances"][0])):
    print(f"{i}. (Distance: {distance:.4f})")
    print(f"   Document: {doc}")
    print(f"   Metadata: {metadata}")
    print()

0. (Distance: 1.2406)
   Document: Deep learning is a subset of machine learning using multi-layered neural networks.
   Metadata: {'topic': 'deep learning', 'category': 'AI'}

1. (Distance: 1.2811)
   Document: Neural networks are inspired by the structure of the human brain.
   Metadata: {'category': 'AI', 'topic': 'neural networks'}

2. (Distance: 1.3072)
   Document: Natural language processing enables computers to understand human language.
   Metadata: {'category': 'AI', 'topic': 'NLP'}



In [ ]:
results = collection.query(query_texts=["Tell me about AI"], 
                           n_results=3)

